# **Explorando IA Generativa com Hugging Face e Google Gemini**



**Construindo um pipeline de geração de texto**

Os pipelines do Hugging Face facilitam o uso de modelos de Machine Learning para várias tarefas. Neste exercício, você vai construir um pipeline de geração de texto usando o modelo gpt2 e personalizar a saída ajustando seus parâmetros.

Fique à vontade para experimentar diferentes prompts no pipeline, como "E se …?", "Como …?", ou qualquer outra ideia criativa que você queira explorar.

In [ ]:
# transformers  → biblioteca principal do Hugging Face para usar modelos de NLP
# torch         → framework de deep learning (necessário para rodar os modelos localmente)
# huggingface_hub → SDK para acessar a API remota do Hugging Face
!pip install transformers torch huggingface_hub -q

In [ ]:
# pipeline → função de alto nível que encapsula modelo + tokenizer + pós-processamento
# Com ela você não precisa gerenciar cada etapa manualmente
from transformers import pipeline
# InferenceClient → cliente para chamar modelos remotamente via API do Hugging Face
# Não baixa o modelo — a inferência acontece nos servidores deles
from huggingface_hub import InferenceClient

**Geração de texto com GPT-2**

In [ ]:
# pipeline() cria um objeto pronto para uso
# task="text-generation" → diz qual tipo de tarefa queremos realizar
# model="openai-community/gpt2" → nome do modelo no Hub do Hugging Face
# Na primeira execução, o modelo (~500MB) será baixado e armazenado em cache
gerador = pipeline(task="text-generation", model="openai-community/gpt2")

In [ ]:
resultados = gerador(
    "What if AI",           # Prompt: texto inicial que o modelo vai continuar
    max_new_tokens=20,      # Limite de tokens NOVOS gerados (não conta o prompt)
    num_return_sequences=5  # Quantas continuações diferentes gerar
)

In [ ]:
# Cada resultado é um dicionário com a chave 'generated_text'
# que contém o prompt original + o texto gerado
for i, r in enumerate(resultados):
    print(f"[Opção {i+1}] {r['generated_text']}")

**Análise de sentimento**

In [ ]:
# Sem especificar model=, o Hugging Face usa o modelo padrão da tarefa
# Para sentiment-analysis: "distilbert-base-uncased-finetuned-sst-2-english"
# Classifica o texto como POSITIVE ou NEGATIVE com uma pontuação de confiança
sentimento = pipeline(task="sentiment-analysis")

In [ ]:
frases = [
    "I love using Hugging Face models!",
    "This is terrible and frustrating.",
]

In [ ]:
for frase in frases:
    r = sentimento(frase)[0]  # [0] porque o retorno é sempre uma lista
    # r['label']  → rótulo previsto: POSITIVE ou NEGATIVE
    # r['score']  → confiança do modelo na previsão (0 a 1)
    print(f"{frase}\n→ {r['label']} ({r['score']:.2%})\n")


**Resumo de texto (sumarização)**

In [ ]:
# ATENÇÃO: versões recentes do transformers removeram a task "summarization"
# Solução: usar task="text2text-generation" que é o nome atual para modelos seq2seq
# facebook/bart-large-cnn → modelo BART treinado para sumarização de notícias
from transformers import BartForConditionalGeneration, BartTokenizer

In [ ]:
# Carrega o tokenizer (converte texto → tokens numéricos) e o modelo separadamente
# Isso dá mais controle do que usar pipeline() para este caso
tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
modelo_resumo = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")

In [ ]:
texto = """
Artificial intelligence is transforming industries around the world.
From healthcare to finance, AI systems are being deployed to automate tasks,
analyze large datasets, and assist decision-making. Machine learning enables
computers to learn from data without being explicitly programmed.
Deep learning has driven breakthroughs in image recognition and natural language processing.
"""

In [ ]:
# Tokeniza o texto de entrada e converte para tensores PyTorch (return_tensors="pt")
# truncation=True garante que textos longos sejam cortados ao limite do modelo
inputs = tokenizer(texto, return_tensors="pt", truncation=True, max_length=1024)

In [ ]:
# Gera o resumo — max_length e min_length controlam o tamanho da saída em tokens
ids_resumo = modelo_resumo.generate(
    inputs["input_ids"],
    max_length=60,   # Teto de tokens no resumo gerado
    min_length=20,   # Piso de tokens — evita resumos trivialmente curtos
    early_stopping=True  # Para a geração assim que o modelo emite o token de fim
)

In [ ]:
# decode() converte os tokens numéricos de volta para texto legível
# skip_special_tokens=True remove tokens internos como <s>, </s>, <pad>
print(tokenizer.decode(ids_resumo[0], skip_special_tokens=True))

**Configurar token (Colab Secrets)**

In [ ]:
# Para usar a API remota do Hugging Face é preciso um token de acesso
# Crie o seu gratuitamente em: huggingface.co/settings/tokens
# No Colab: clique no ícone 🔑 (barra lateral) > adicione o secret "HF_TOKEN"
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')  # Lê o token dos Secrets do Colab com segurança
print("✅ Token ok!" if HF_TOKEN else "❌ Adicione HF_TOKEN nos Secrets do Colab (ícone 🔑)")

**Geração de texto via API**

In [ ]:
MODELO = "meta-llama/Llama-3.1-8B-Instruct"

In [ ]:
cliente = InferenceClient(model=MODELO, token=HF_TOKEN, provider="auto")

In [ ]:
resposta = cliente.chat_completion(
    messages=[
        {"role": "user", "content": "Explique inteligência artificial em uma frase simples:"}
    ],
    max_tokens=80,
    temperature=0.7  # 0 = determinístico, 1 = mais criativo
)

In [ ]:
print(resposta.choices[0].message.content)

**Chat com modelo (formato conversa)**

In [ ]:
cliente_chat = InferenceClient(model=MODELO, token=HF_TOKEN, provider="auto")

In [ ]:
mensagens = [
    {"role": "system", "content": "Você é um assistente de tecnologia. Responda de forma clara e objetiva."},
    {"role": "user",   "content": "O que são Large Language Models (LLMs)?"}
]

In [ ]:
resposta_chat = cliente_chat.chat_completion(
    messages=mensagens,
    max_tokens=150
)

In [ ]:
print(resposta_chat.choices[0].message.content)

In [ ]:
# ============================================================
# EXERCÍCIO — Chat interativo com histórico
# ============================================================

# Objetivo: criar um chat em loop onde o aluno digita mensagens
# e o modelo responde, mantendo o histórico da conversa.
#
# Passos:
#   1. Defina uma mensagem de system com a personalidade do assistente
#   2. Crie um loop que:
#      a. leia a entrada do usuário com input()
#      b. adicione a mensagem ao histórico (lista de dicionários)
#      c. envie o histórico para o modelo
#      d. imprima a resposta
#      e. adicione a resposta ao histórico (role "assistant")
#      f. encerre com "sair"

In [ ]:
cliente_ex = InferenceClient(model=MODELO, token=HF_TOKEN, provider="auto")

In [ ]:
historico = [
    {"role": "system", "content": "Você é um assistente prestativo. Responda sempre em português."}
]

In [ ]:
print("Chat iniciado! Digite 'sair' para encerrar.\n")

while True:
    entrada = input("Você: ")

    if entrada.lower() == "sair":
        print("Encerrando chat.")
        break

    historico.append({"role": "user", "content": entrada})

    resposta_ex = cliente_ex.chat_completion(
        messages=historico,
        max_tokens=200
    )

    resposta_texto = resposta_ex.choices[0].message.content

    historico.append({"role": "assistant", "content": resposta_texto})

    print(f"\nAssistente: {resposta_texto}\n")

**Interface web com Gradio**

In [ ]:
# Gradio gera uma interface web automaticamente a partir de uma função Python
# Ao rodar, abre no navegador em http://localhost:7860
# No Colab, gera um link público automaticamente
import gradio as gr

In [ ]:
cliente_gradio = InferenceClient(model=MODELO, token=HF_TOKEN, provider="auto")

In [ ]:
def responder(mensagem, historico):
    # historico é uma lista de tuplas (usuario, assistente) gerenciada pelo Gradio
    mensagens = [{"role": "system", "content": "Você é um assistente prestativo. Responda em português."}]

    for humano, assistente in historico:
        mensagens.append({"role": "user",      "content": humano})
        mensagens.append({"role": "assistant", "content": assistente})

    mensagens.append({"role": "user", "content": mensagem})

    resposta = cliente_gradio.chat_completion(
        messages=mensagens,
        max_tokens=200
    )

    return resposta.choices[0].message.content

In [ ]:
# ChatInterface monta automaticamente a UI de chat com histórico
gr.ChatInterface(
    fn=responder,
    title="Chat com Hugging Face",
    description="Digite sua mensagem e pressione Enter"
).launch()

**Chatbot com Google Gemini**

In [ ]:
# Obtenha sua chave em: aistudio.google.com/apikey
# No Colab: adicione o secret "GEMINI_API_KEY" (ícone 🔑)
from google.colab import userdata
from google import genai
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)
MODELO_GEMINI = "gemini-2.5-flash-lite"  # free tier: 1000 req/dia

In [ ]:
chat_gemini = cliente_gemini.chats.create(model=MODELO_GEMINI)

print(f"Chat Gemini iniciado! Digite 'sair' para encerrar.\n")

while True:
    entrada = input("Você: ")

    if entrada.lower() == "sair":
        print("Encerrando chat.")
        break

    try:
        resposta = chat_gemini.send_message(entrada)
        print(f"\nGemini: {resposta.text}\n")

    except Exception as e:
        print(f"\nErro: {e}\n")

**Integrando IA generativa em apps e sites**

In [ ]:
# CONCEITO:
# Modelos de IA são consumidos como qualquer outra API REST.
# O frontend (site, app mobile) envia texto via HTTP POST
# e recebe o texto gerado como resposta JSON.
#
# FLUXO TÍPICO:
#
#   [Frontend]  →  HTTP POST /chat  →  [Backend Python]  →  HuggingFace API
#   React/Vue       { "msg": "..." }    FastAPI/Flask         modelo LLM
#   Flutter/RN                              ↓
#   HTML puro                         retorna JSON
#                                      { "resposta": "..." }
#
# POR QUE TER UM BACKEND?
#   - O HF_TOKEN nunca pode ficar exposto no frontend (segurança)
#   - O backend centraliza a lógica, histórico e regras de negócio
#   - Permite trocar o modelo sem mudar o frontend

In [ ]:
# ------------------------------------------------------------------
# EXEMPLO: servidor FastAPI que expõe o modelo como endpoint REST
# ------------------------------------------------------------------
# pip install fastapi uvicorn
#
# Para rodar: uvicorn nome_do_arquivo:app --reload
# O frontend chama: POST http://localhost:8000/chat
#                   Body: { "mensagem": "Olá!" }

from fastapi import FastAPI
from pydantic import BaseModel
from huggingface_hub import InferenceClient

app    = FastAPI()
client = InferenceClient(model=MODELO, token=HF_TOKEN, provider="auto")

class Pergunta(BaseModel):
    mensagem: str

@app.post("/chat")
def chat(pergunta: Pergunta):
    resposta = client.chat_completion(
        messages=[{"role": "user", "content": pergunta.mensagem}],
        max_tokens=200
    )
    return {"resposta": resposta.choices[0].message.content}

In [ ]:
# ------------------------------------------------------------------
# EXEMPLO: como o frontend chama esse endpoint (JavaScript)
# ------------------------------------------------------------------
#
# fetch("http://localhost:8000/chat", {
#   method: "POST",
#   headers: { "Content-Type": "application/json" },
#   body: JSON.stringify({ mensagem: "Explique machine learning" })
# })
# .then(r => r.json())
# .then(data => console.log(data.resposta))
#
# ------------------------------------------------------------------
# RESUMO: qual tecnologia usar?
# ------------------------------------------------------------------
#
#   Gradio / Streamlit  → protótipo rápido, sem frontend separado
#   FastAPI + React     → app web completo
#   FastAPI + Flutter   → app mobile
#   FastAPI + APEX      → integração com sistemas Oracle